# Additional File Concepts — CSV, JSON, Atomic Writes, and Real Work 🔐

## Goal

Build safe structured-data and large-file habits beyond basic `open`.

## CSV and JSON

In [1]:
import csv,json,tempfile
from pathlib import Path
records=[{'name':'Maya','score':95},{'name':'Leo','score':88}]
with tempfile.TemporaryDirectory() as folder:
    root=Path(folder); csv_path=root/'scores.csv'; json_path=root/'scores.json'
    with csv_path.open('w',encoding='utf-8',newline='') as file:
        writer=csv.DictWriter(file,fieldnames=['name','score']); writer.writeheader(); writer.writerows(records)
    json_path.write_text(json.dumps(records,indent=2),encoding='utf-8')
    with csv_path.open(encoding='utf-8',newline='') as file: print(list(csv.DictReader(file)))
    print(json.loads(json_path.read_text(encoding='utf-8')))

[{'name': 'Maya', 'score': '95'}, {'name': 'Leo', 'score': '88'}]
[{'name': 'Maya', 'score': 95}, {'name': 'Leo', 'score': 88}]


## Atomic-style replacement and backups

For important updates: write a temporary neighbor, flush/close it, then replace the target. A crash is less likely to leave a half-written file. Backups and OS-level atomic guarantees still depend on filesystem/platform.

In [2]:
import os
with tempfile.TemporaryDirectory() as folder:
    target=Path(folder)/'config.json'; target.write_text('{"version":1}',encoding='utf-8')
    temporary=target.with_suffix('.tmp'); temporary.write_text('{"version":2}',encoding='utf-8')
    os.replace(temporary,target)
    print(target.read_text(encoding='utf-8'))

{"version":2}


## Stream large files and process errors per line

Do not load everything when one record at a time is enough. Track line numbers so bad data can be explained.

In [3]:
lines=['10\n','oops\n','30\n']
valid=[]; problems=[]
for number,line in enumerate(lines,1):
    try: valid.append(int(line))
    except ValueError as error: problems.append((number,str(error)))
print(valid,problems)

[10, 30] [(2, "invalid literal for int() with base 10: 'oops\\n'")]


## Other important concepts

- `io.StringIO`/`BytesIO`: in-memory file-like objects for tests.
- `tempfile`: safe temporary files/directories.
- `shutil`: high-level copy/move/archive.
- `gzip`: compressed streams.
- `pickle`: Python-specific binary serialization—**never unpickle untrusted data**.
- file locking/concurrent writers require deliberate coordination.

## Key concepts summary

Structured files need schemas and validation; safe updates need temporary outputs, explicit encoding, and careful error reporting.

## Important syntax and quick revision cheat sheet

| Need | Syntax |
|---|---|
| CSV rows | `csv.DictReader/DictWriter` |
| JSON | `json.load/dump` |
| Temporary work | `tempfile` |
| Atomic replace | `os.replace(temp,target)` |
| In-memory text file | `io.StringIO` |

## Common mistakes and interview tips

- Never unpickle untrusted bytes.
- JSON keys become strings and not every Python object is serializable.
- Validate CSV headers and conversions.
- Coordinate concurrent writers.
- Preserve originals until replacement is verified.

**Revision habit:** explain what each line does, predict the result, run it, and test one edge case.